# Hands-on Task: Chain, Validate, Route

Runs a summarize-then-classify LangChain pipeline with validation between steps, and sends a real notification when a review comes back negative.

## Setup
Store your key in Colab **Secrets** (key icon, left sidebar) as `OPENAI_API_KEY`.

In [ ]:
!pip install -q langchain langchain-openai langchain-core

In [ ]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", api_key=userdata.get("OPENAI_API_KEY"))

In [ ]:
!pip install -q requests

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
summarize_prompt = ChatPromptTemplate.from_template("Summarize this review in one sentence:\n{review}")
classify_prompt = ChatPromptTemplate.from_template("Classify this as positive or negative. Respond with only one word:\n{summary}")
summarize_chain = summarize_prompt | llm | StrOutputParser()
classify_chain = classify_prompt | llm | StrOutputParser()

In [ ]:
import requests
NTFY_TOPIC = "ojasa-mirai-agent-demo-CHANGE-ME"
def notify_course_team(summary):
    r = requests.post(f"https://ntfy.sh/{NTFY_TOPIC}", data=f"Negative review flagged: {summary}".encode("utf-8"))
    return r.status_code

In [ ]:
def run_pipeline(review):
    summary = summarize_chain.invoke({"review": review})
    if len(summary.split()) < 3:
        return f"Broken link: summary too short ({summary!r})"
    classification = classify_chain.invoke({"summary": summary})
    if "negative" in classification.lower():
        status = notify_course_team(summary)
        return f"{classification} -> notified course team (status {status})"
    return f"{classification} (summary: {summary})"

In [ ]:
review = ("I finished the AI Engineer course last month. The RAG and tool-calling modules "
          "were incredibly practical, and the mentor sessions kept me on track every week. "
          "Hands down the best course I've taken.")
negative_review = ("I enrolled expecting live mentorship but most sessions were cancelled or "
                    "rescheduled last minute. The content itself was fine, but I can't recommend "
                    "it because of the support issues.")
for r in [review, negative_review, "asdkfj laksjdf"]:
    print(run_pipeline(r))

## How to Extend
Open https://ntfy.sh/your-topic in another tab before running, to watch the notification arrive live for the negative review.

## Done When
All three inputs behave correctly: the clean review classifies, the negative one triggers a real notification, and the broken one is caught by validation.